# 🏠 Melbourne House Price Prediction
### A Complete Machine Learning Project

[![Python](https://img.shields.io/badge/Python-3.10+-blue.svg)](https://www.python.org)
[![scikit-learn](https://img.shields.io/badge/scikit--learn-1.2+-orange.svg)](https://scikit-learn.org)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)

---

## 🎯 Objective

Build and compare machine learning models to **predict house sale prices** in Melbourne, Australia.  
We cover the full ML workflow: data exploration → cleaning → feature selection → model training → evaluation → prediction.

## 📦 Dataset

- **Source**: [Melbourne Housing Snapshot — Kaggle](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot)
- **Size**: 13,580 properties × 21 features
- **Target**: `Price` (sale price in AUD)

---

## Table of Contents

1. [Setup & Imports](#1-setup)
2. [Data Loading & Exploration](#2-exploration)
3. [Data Cleaning](#3-cleaning)
4. [Train / Test Split](#4-split)
5. [Model Training & Comparison](#5-models)
6. [Best Model Deep Dive](#6-best)
7. [Feature Importance](#7-importance)
8. [Predict a New House](#8-predict)
9. [Key Takeaways](#9-takeaways)


## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Plotting style
plt.rcParams.update({
    "figure.facecolor": "#0f172a",
    "axes.facecolor":   "#1e293b",
    "text.color":       "#e2e8f0",
    "axes.labelcolor":  "#e2e8f0",
    "xtick.color":      "#e2e8f0",
    "ytick.color":      "#e2e8f0",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.bottom": False,
    "axes.spines.left":   False,
})
COLORS = ["#38bdf8", "#818cf8", "#34d399", "#f59e0b", "#f87171"]

print("✅ All libraries loaded")

## 2. Data Loading & Exploration <a id='2-exploration'></a>

The first step in any ML project is to **understand the data** before touching a model.  
We check: the shape, column types, missing values, and the distribution of our target variable.


In [ ]:
df = pd.read_csv("data/melb_data.csv")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# Data types and null counts at a glance
df.info()

In [ ]:
# Summary stats for key numeric columns
df[["Price", "Rooms", "Distance", "Bathroom", "Landsize", "BuildingArea"]].describe().round(0)

In [ ]:
# Missing values — important to know before building a pipeline
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:")
for col, n in missing.items():
    print(f"  {col:<20} {n:>5} ({n/len(df)*100:.1f}%)")

In [ ]:
# Price distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df["Price"].dropna() / 1e6, bins=60, color="#38bdf8", edgecolor="none", alpha=0.85)
median_m = df["Price"].median() / 1e6
ax.axvline(median_m, color="#f59e0b", linewidth=1.5, label=f"Median: ${median_m:.2f}M")
ax.set_title("Distribution of House Prices in Melbourne", color="#e2e8f0", pad=12)
ax.set_xlabel("Price (Millions AUD)")
ax.set_ylabel("Number of Houses")
ax.legend(facecolor="#1e293b", labelcolor="#e2e8f0")
plt.tight_layout()
plt.savefig("outputs/01_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations:**
- Prices range from ~\$85k to \$9M with a **median around \$900k**
- The distribution is **right-skewed** — a small number of luxury homes pull the mean up
- This is typical for real estate data


## 3. Data Cleaning <a id='3-cleaning'></a>

**Strategy:**
- Select only numeric features relevant to pricing
- Drop rows where the **target (Price)** is missing
- Handle missing values in features automatically via `SimpleImputer` inside the pipeline  
  *(filling with the column median — robust to outliers)*


In [ ]:
FEATURES = [
    'Rooms', 'Distance', 'Bedroom2', 'Bathroom', 'Car',
    'Landsize', 'BuildingArea', 'YearBuilt', 'Propertycount'
]
TARGET = 'Price'

df_clean = df[FEATURES + [TARGET]].dropna(subset=[TARGET])

X = df_clean[FEATURES]
y = df_clean[TARGET]

print(f"Features : {FEATURES}")
print(f"Target   : {TARGET}")
print(f"Shape    : {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"\nRemaining NaNs per feature:")
print(X.isnull().sum())

## 4. Train / Test Split <a id='4-split'></a>

> **Golden Rule:** The model must **never see the test data** during training.  
> The test set is our "final exam" — it tells us how well the model works on *unseen* houses.

We use an **80 / 20** split.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set : {len(X_train):,} houses  ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set     : {len(X_test):,}  houses  ({len(X_test)/len(X)*100:.0f}%)")

## 5. Model Training & Comparison <a id='5-models'></a>

We compare **5 models** of increasing complexity.  
Each model is wrapped in a **Pipeline** that handles preprocessing automatically.

| Step | What it does |
|------|-------------|
| `SimpleImputer` | Fills missing values with the column **median** |
| `StandardScaler` | Normalises features to mean=0, std=1 |
| `model` | The regressor |

### Evaluation Metrics

| Metric | Formula | Meaning |
|--------|---------|---------|
| **R²** | 1 - SS_res/SS_tot | % of price variance explained. 1.0 = perfect |
| **MAE** | mean(\|y - ŷ\|) | Average prediction error in $ |
| **RMSE** | √mean((y - ŷ)²) | Like MAE but penalises large errors more |
| **CV R²** | mean R² over 5 folds | Robustness: does the model generalise? |


In [ ]:
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   model),
    ])

models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=10)),
    "Decision Tree":     make_pipeline(DecisionTreeRegressor(max_depth=6, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    "Gradient Boosting": make_pipeline(GradientBoostingRegressor(n_estimators=100, random_state=42)),
}

results = {}
rows    = []

print(f"{'Model':<22} {'MAE ($)':>12} {'RMSE ($)':>12} {'R²':>8} {'CV R²':>8}")
print("-" * 68)

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2").mean()

    results[name] = {"pipeline": pipeline, "y_pred": y_pred,
                     "mae": mae, "rmse": rmse, "r2": r2, "cv_r2": cv_r2}
    rows.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2, "CV_R2": cv_r2})
    print(f"{name:<22} {mae:>12,.0f} {rmse:>12,.0f} {r2:>8.3f} {cv_r2:>8.3f}")

summary = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)

In [ ]:
# Visual comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

names    = summary["Model"].tolist()
r2_vals  = summary["R2"].tolist()
mae_vals = (summary["MAE"] / 1000).tolist()
cols     = COLORS[:len(names)]

# R²
bars = ax1.barh(names, r2_vals, color=cols, height=0.5, edgecolor="none")
ax1.axvline(0.8, color="#f59e0b", linestyle="--", linewidth=1.2, label="Good threshold (0.8)")
for bar, v in zip(bars, r2_vals):
    ax1.text(v + 0.005, bar.get_y() + bar.get_height()/2,
             f"{v:.3f}", va="center", color="#e2e8f0", fontsize=9, fontweight="bold")
ax1.set_xlim(0, 1.05)
ax1.set_title("R² Score  (higher = better)", pad=10)
ax1.set_xlabel("R²")
ax1.legend(facecolor="#1e293b", labelcolor="#e2e8f0", fontsize=8)

# MAE
bars2 = ax2.barh(names, mae_vals, color=cols, height=0.5, edgecolor="none")
for bar, v in zip(bars2, mae_vals):
    ax2.text(v + 2, bar.get_y() + bar.get_height()/2,
             f"${v:.0f}k", va="center", color="#e2e8f0", fontsize=9, fontweight="bold")
ax2.set_title("Mean Absolute Error  (lower = better)", pad=10)
ax2.set_xlabel("MAE (k$)")

plt.suptitle("Model Comparison — Melbourne House Price Prediction",
             color="#e2e8f0", fontsize=13, y=1.02, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/02_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Best Model Deep Dive <a id='6-best'></a>

The **Random Forest** achieves the best R² on the test set.  
Let's inspect its predictions vs. reality.


In [ ]:
best_name = max(results, key=lambda k: results[k]["r2"])
best = results[best_name]

print(f"🏆 Best model : {best_name}")
print(f"   R²    = {best['r2']:.3f}")
print(f"   MAE   = ${best['mae']:,.0f}")
print(f"   RMSE  = ${best['rmse']:,.0f}")
print(f"   CV R² = {best['cv_r2']:.3f}")

In [ ]:
# Actual vs Predicted scatter plot
fig, ax = plt.subplots(figsize=(7, 6))

sample = min(600, len(y_test))
idx = np.random.choice(len(y_test), sample, replace=False)
yt = np.array(y_test)[idx] / 1e6
yp = best["y_pred"][idx] / 1e6

ax.scatter(yt, yp, alpha=0.35, s=14, color="#38bdf8", label="Houses")
max_val = max(yt.max(), yp.max())
ax.plot([0, max_val], [0, max_val], color="#f59e0b", linewidth=1.5, label="Perfect prediction")
ax.set_title(f"Actual vs Predicted — {best_name}\n(points on the diagonal = perfect)", pad=10)
ax.set_xlabel("Actual Price (M$)")
ax.set_ylabel("Predicted Price (M$)")
ax.legend(facecolor="#1e293b", labelcolor="#e2e8f0")
plt.tight_layout()
plt.savefig("outputs/03_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the chart:**
- Points close to the diagonal line = accurate predictions
- Points **above** the line = model overestimates the price
- Points **below** the line = model underestimates
- The spread widens at higher prices — luxury homes are harder to predict (less data, more unique)


## 7. Feature Importance <a id='7-importance'></a>

One of the key advantages of tree-based models is **interpretability** — we can see exactly which features drive predictions.


In [ ]:
rf_model = best["pipeline"].named_steps["model"]
importances = rf_model.feature_importances_
sorted_idx  = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(FEATURES)))
ax.barh([FEATURES[i] for i in sorted_idx], importances[sorted_idx],
        color=[colors[i] for i in range(len(sorted_idx))], edgecolor="none")
ax.set_title("Feature Importance — Random Forest\n(which variables influence price the most?)", pad=10)
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig("outputs/04_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 3 most important features:")
for i in reversed(sorted_idx[-3:]):
    print(f"  {FEATURES[i]:<20} {importances[i]:.3f}")

**Key findings:**
- `Distance` from the CBD is the **single most important driver** of Melbourne house prices
- `BuildingArea` and `Rooms` also carry significant weight
- `Car` spaces and `Propertycount` have the least predictive power in this dataset


## 8. Predict a New House <a id='8-predict'></a>

Now we can use our trained model to estimate the price of any house.


In [ ]:
# Define a new house
new_house = pd.DataFrame([{
    "Rooms":         3,
    "Distance":      10,
    "Bedroom2":      3,
    "Bathroom":      1,
    "Car":           1,
    "Landsize":      500,
    "BuildingArea":  120,
    "YearBuilt":     1990,
    "Propertycount": 4000,
}])

print("🏠 House features:")
for col, val in new_house.iloc[0].items():
    print(f"   {col:<20}: {val}")

price = best["pipeline"].predict(new_house)[0]
print(f"\n💰 Estimated price ({best_name}): ${price:,.0f} AUD")

In [ ]:
# Try different house configurations and see how price changes
scenarios = [
    {"label": "Small unit (1 room, 3km)",  "Rooms": 1, "Distance": 3,  "Bedroom2": 1, "Bathroom": 1, "Car": 0, "Landsize": 150, "BuildingArea": 50,  "YearBuilt": 2010, "Propertycount": 8000},
    {"label": "Family home (3 rooms, 10km)","Rooms": 3, "Distance": 10, "Bedroom2": 3, "Bathroom": 1, "Car": 1, "Landsize": 500, "BuildingArea": 120, "YearBuilt": 1990, "Propertycount": 4000},
    {"label": "Large house (5 rooms, 20km)","Rooms": 5, "Distance": 20, "Bedroom2": 5, "Bathroom": 3, "Car": 2, "Landsize": 900, "BuildingArea": 250, "YearBuilt": 2000, "Propertycount": 3000},
    {"label": "Luxury (6 rooms, 5km)",      "Rooms": 6, "Distance": 5,  "Bedroom2": 6, "Bathroom": 3, "Car": 3, "Landsize": 700, "BuildingArea": 350, "YearBuilt": 2015, "Propertycount": 5000},
]

print(f"{'Scenario':<35} {'Estimated Price':>18}")
print("-" * 55)
for s in scenarios:
    label = s.pop("label")
    p = best["pipeline"].predict(pd.DataFrame([s]))[0]
    print(f"{label:<35} ${p:>16,.0f}")

## 9. Key Takeaways <a id='9-takeaways'></a>

### Results Summary

| Model | R² | MAE | Notes |
|-------|----|-----|-------|
| Linear Regression | ~0.44 | ~$328k | Too simple for this data |
| Ridge | ~0.44 | ~$328k | Marginal improvement over Linear |
| Decision Tree | ~0.51 | ~$297k | Better, but overfits easily |
| **Random Forest** | **~0.73** | **~$209k** | ✅ Best overall — robust and accurate |
| Gradient Boosting | ~0.67 | ~$247k | Strong, slightly below RF here |

### What I learned

1. **Distance from CBD** is the #1 predictor of Melbourne house prices
2. **Ensemble methods** (Random Forest, Gradient Boosting) significantly outperform linear models
3. **Cross-validation** is essential — a good test R² alone can be misleading
4. **Missing data** (BuildingArea: 47% missing, YearBuilt: 40%) is the main data quality challenge

### Possible Improvements

- Add **categorical features** (Suburb, Type, CouncilArea) using One-Hot Encoding
- Use **log-transform** on Price to handle the right-skewed distribution
- Try **XGBoost** or **LightGBM** for better performance
- Add **hyperparameter tuning** with `GridSearchCV` or `RandomizedSearchCV`
- Include **geographic coordinates** (Lat/Long) as spatial features

---
*Project by — built with scikit-learn, pandas, and matplotlib*
